# Verification: Dolan et al. (2015), NHEJ repair and p53/p21 early senescence (network-free)

`models/p53_nhej_dolan2015` is the integrated single-cell stochastic model of Dolan et al. (2015),
PLoS Comput Biol 11:e1004246: non-homologous end joining of DNA double-strand breaks at fifty
independent loci, ATM/p53/Mdm2 damage signalling, and a p21-driven early-senescence switch. It has
74 molecule types, 2355 reaction rules (121 with function rate laws) and about 2,300 particles, and
it is simulated network-free, as the authors did.

**What is run.** Every simulation below is stock BioNetGen 2.9.3 `BNG2.pl` driving its NFsim 1.14.3,
on the committed `.bngl` **unchanged**: a two-line wrapper reads the file with
`readFile({skip_actions=>1})`, so the model BNG2.pl parses is the committed file byte for byte, and
then runs the file's own `simulate` action with a stated seed added. No bngsim is involved. (The
mean-field ODE comparison in `run_dolan2015.py independent` runs on an edited copy with DNA damage
switched off and its `agreement` mode runs through bngsim; neither is used here.) Each replicate has
its own seed, so the ensemble is a set of independent cells.

**Level 1, exact theory on the full model.** Two parts of the network have a closed form that holds
with damage, repair and signalling all running, and both are transcribed from the paper's reaction
list (S1 Table) and rate constants (S3 Table), not from the generated XML:

* p53 mRNA is an immigration-death process (`P() -> P() + p53_mRNA()`, `p53_mRNA() -> 0`), so its
  mean and variance are known at every time;
* ROS is made at `kROS` and by phospho-p38 at `kROSgen`, and degraded at `kdROS`, so
  `E[ROS - (kROSgen/kdROS) p38p] = kROS/kdROS` at quasi-steady state (the ROS consumed by the
  damage reactions is a 1.2e-4 relative correction, bounded below).

**Level 2, what the paper states for non-irradiated cells.** The paper simulated non-irradiated
cells for 30 hours (Methods, *Simulation*) and describes them in *Stochastic simulations of cell
fate* and in the discussion of Fig. 2. Its statements are qualitative except two: that *none* of
the simulated cells became senescent, and that a 10-fold increase in background ROS sends 32% of
cells into early senescence after 30 hours. Fig. 4 shows two representative single runs, which a
stochastic model cannot be held to trajectory by trajectory, so each statement is asserted as an
ensemble statistic with its tolerance stated in the cell. Where the paper's words are qualitative
("approximately constant", "rare", "almost all"), the cell says how they are read; those readings
and every tolerance were fixed before the ensemble was run.

In [1]:
import hashlib, math, os, re, shutil, subprocess, sys, tempfile
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
from scipy.stats import norm
import matplotlib.pyplot as plt

HERE = Path.cwd()
STEM = "p53_nhej_dolan2015"
MODEL = HERE / f"{STEM}.bngl"
TEXT = MODEL.read_text()
sys.path.insert(0, str(HERE.parents[1] / "skills/curate-model/scripts"))
from verification_figure import verdict_footer

WORKERS = int(os.environ.get("BNGLM_WORKERS", "2"))   # concurrent BNG2.pl runs


def find_bionetgen():
    """BNG2.pl from $BNG2_PL or $BNGPATH, then BNG2.pl on PATH, then the `bionetgen` CLI."""
    for cand in (os.environ.get("BNG2_PL"),
                 str(Path(os.environ["BNGPATH"]) / "BNG2.pl") if os.environ.get("BNGPATH") else None,
                 shutil.which("BNG2.pl")):
        if cand and Path(cand).is_file():
            return str(Path(cand).absolute()), False
    cli = shutil.which("bionetgen")
    if cli:
        return cli, True
    raise FileNotFoundError("BioNetGen not found: set BNG2_PL (or BNGPATH) to a BioNetGen 2.9.3 "
                            "BNG2.pl, or put the bionetgen CLI on PATH.")


BNG, USE_CLI = find_bionetgen()
print("BioNetGen:", BNG)
print(f"model: {MODEL.name}  sha256 {hashlib.sha256(MODEL.read_bytes()).hexdigest()}")
print("concurrent BNG2.pl runs:", WORKERS)

# Rate constants and seed counts are read from the committed file, never retyped.
_block = lambda name: re.split(rf"(?m)^begin {name}\s*$", TEXT)[1].split(f"end {name}")[0]
P = {m.group(1): float(m.group(2))
     for m in re.finditer(r"(?m)^\s*(\w+)\s+([0-9.eE+-]+)\s*(?:#|$)", _block("parameters"))}
SEED_COUNT = {m.group(1): float(m.group(2))
              for m in re.finditer(r"(?m)^\s*(\S+)\s+([0-9.eE+-]+)\s*(?:#|$)", _block("seed species"))}
P21_HIGH = float(re.search(r"kplus\(\)\s*=\s*if\(p21>([0-9.]+)", TEXT).group(1))

# The file's own simulate action (its actions block with comments and line continuations removed).
_act = re.sub(r"(?m)#.*$", "", TEXT.split("\nend model", 1)[1]).replace("\\\n", " ")
(FILE_ACTION,) = re.findall(r"simulate\s*\(\{.*?\}\)", _act, re.S)
FILE_ACTION = re.sub(r"\s+", "", FILE_ACTION)
print("the file's own action:", FILE_ACTION)

TMP = Path(tempfile.mkdtemp(prefix="nfruns_", dir=HERE))   # removed at the end


def action(seed, **override):
    """The file's simulate action with a seed added and, optionally, t_end / n_steps replaced."""
    a = FILE_ACTION
    for k, v in override.items():
        a, n = re.subn(rf"{k}=>[^,}}]+", f"{k}=>{v}", a)
        assert n == 1, (k, a)
    return a[:-2] + f",seed=>{seed}}})"


def run_bng(seed, set_params=(), **override):
    """One BNG2.pl NFsim run of the committed file. The wrapper reads it unchanged with
    readFile({skip_actions=>1}), applies any setParameter, then runs the file's own action."""
    d = Path(tempfile.mkdtemp(prefix=f"s{seed}_", dir=TMP))
    w = d / f"{STEM}.bngl"     # the same stem, so the XML model id and the output names match
    w.write_text("begin actions\n"
                 f'readFile({{file=>"{MODEL}",skip_actions=>1}})\n'
                 + "".join(f'setParameter("{k}",{v})\n' for k, v in set_params)
                 + action(seed, **override) + "\nend actions\n")
    cmd = ([BNG, "run", "-i", str(w), "-o", str(d)] if USE_CLI
           else ["perl", BNG, "--outdir", str(d), str(w)])
    r = subprocess.run(cmd, capture_output=True, text=True, cwd=d)
    assert r.returncode == 0, r.stdout[-3000:] + r.stderr[-3000:]
    g = d / f"{STEM}_nfr.gdat"
    hdr = g.open().readline().lstrip("#").split()
    return d, r.stdout, hdr, np.loadtxt(g)

BioNetGen: /vast/home/wish/wt/scratch/bng-loader/BioNetGen-2.9.3/BNG2.pl
model: p53_nhej_dolan2015.bngl  sha256 896e0421d215236f9ffae1377c9799810b39995bc0cd1a06e4c9faba878a880a
concurrent BNG2.pl runs: 64
the file's own action: simulate({method=>"nf",suffix=>"nfr",t_start=>0,t_end=>2400,n_steps=>240,gml=>2147483647})


## The fresh run against the committed reference, and what the file is

`reference/p53_nhej_dolan2015_nfr.gdat` is one cell run with the file's own action at seed 2015
(the action block itself carries no seed). The house rule of `skills/curate-model/SKILL.md` holds
it to the same bar as a deterministic output: every value within a relative error of 1e-3 or an
absolute error of `1e-6 x max|column|`, and the XML NFsim reads must match the committed one
structurally (comments and whitespace ignored).

In [2]:
def xml_signature(path):
    return [l.strip() for l in Path(path).read_text().splitlines()
            if l.strip() and not l.strip().startswith("<!--")]

d_ref, LOG_REF, hdr, fresh = run_bng(2015)
REFDIR = HERE / "reference"
ref_path = REFDIR / f"{STEM}_nfr.gdat"
ref_hdr = ref_path.open().readline().lstrip("#").split()
ref = np.loadtxt(ref_path)
assert hdr == ref_hdr and fresh.shape == ref.shape, (fresh.shape, ref.shape)
atol = 1e-6 * np.maximum(np.abs(ref).max(axis=0), 1e-300)
ok = (np.abs(fresh - ref) <= 1e-3 * np.abs(ref)) | (np.abs(fresh - ref) <= atol)
assert ok.all(), "the fresh seed-2015 run does not reproduce the committed reference .gdat"
print(f"seed 2015, {FILE_ACTION}: {fresh.shape[0]} outputs x {fresh.shape[1]} columns, "
      f"max |fresh - committed| = {np.abs(fresh - ref).max():.1e}")
assert xml_signature(d_ref / f"{STEM}_nfr.xml") == xml_signature(REFDIR / f"{STEM}.xml")
print(f"{STEM}_nfr.xml: structural match with reference/{STEM}.xml")

# What the file is, from BNG2.pl's own log and the XML it wrote.
READ = {k: int(v) for v, k in re.findall(
    r"Read (\d+) (parameters|molecule types|species|observable\(s\)|functions|reaction rule\(s\))",
    LOG_REF)}
xml = (d_ref / f"{STEM}_nfr.xml").read_text()
N_FUNC_RL = len(re.findall(r'<RateLaw[^>]*type="Function"', xml))
PARTICLES = sum(float(c) for c in re.findall(r'<Species id="[^"]*"\s+concentration="([^"]+)"', xml))
NFSIM_CMD = re.search(r"full command: (.*)", LOG_REF).group(1)
print(f"\n{READ['molecule types']} molecule types, {READ['reaction rule(s)']} reaction rules "
      f"({N_FUNC_RL} with function rate laws), {READ['functions']} functions, "
      f"{READ['observable(s)']} observables; {PARTICLES:.0f} particles at t = 0")
print("NFsim command:", NFSIM_CMD.replace(str(d_ref) + "/", ""))
assert (READ["molecule types"], READ["reaction rule(s)"], N_FUNC_RL) == (74, 2355, 121)
assert "-bscb" not in NFSIM_CMD and "-utl" not in NFSIM_CMD
shutil.rmtree(d_ref)

seed 2015, simulate({method=>"nf",suffix=>"nfr",t_start=>0,t_end=>2400,n_steps=>240,gml=>2147483647}): 241 outputs x 177 columns, max |fresh - committed| = 0.0e+00
p53_nhej_dolan2015_nfr.xml: structural match with reference/p53_nhej_dolan2015.xml

74 molecule types, 2355 reaction rules (121 with function rate laws), 9 functions, 176 observables; 2341 particles at t = 0
NFsim command: /vast/home/wish/wt/scratch/bng-loader/BioNetGen-2.9.3/bin/NFsim -xml p53_nhej_dolan2015_nfr.xml -o p53_nhej_dolan2015_nfr.gdat -sim 2400 -oSteps 240 -gml 2147483647 -ss p53_nhej_dolan2015_nfr.species -seed 2015 -cb


## The ensemble

The published SI file runs fifty cells with `simulate_nf({t_end=>2400, n_steps=>2400})`: 40 h at
one output per minute. The ensemble below is that protocol, through the file's own action with only
`n_steps` raised from the committed 240 to the published 2400 (output resolution only) and a
distinct seed per cell, `20150001 ... 20150000 + N`. The paper's statements are about 30 h, so they
are judged on 0-1800 min; the senescence count is also printed for the whole 40 h.

N = 300 cells: the senescence statement is an exact zero, and 300 cells give 95% power to catch a
per-cell senescence probability of 1%; the binomial and theory checks below carry their standard
errors with them. Each run takes a few minutes, so the notebook runs `$BNGLM_WORKERS` at a time.

In [3]:
N_CELLS, T_END, DT = 300, 2400, 1
SEEDS = [20150000 + i for i in range(1, N_CELLS + 1)]
OBS = ["ROS", "p38p", "p53", "p21", "p53_mRNA", "Senescent_Counter"]
t = np.arange(0, T_END + DT, DT, dtype=float)


def focus_runs(foci):
    """Contiguous runs of a damage focus at each locus: (onset min, duration min, censored)."""
    runs = []
    for f in foci:
        x = np.diff(np.concatenate([[0], (f > 0).astype(int), [0]]))
        for s, e in zip(np.flatnonzero(x == 1), np.flatnonzero(x == -1)):
            runs.append((s * DT, (e - s) * DT, float(e == len(f))))
    return np.array(runs, dtype=float).reshape(-1, 3)


def cell(seed):
    d, _, hdr, a = run_bng(seed, n_steps=T_END // DT)
    col = {h: i for i, h in enumerate(hdr)}
    assert np.allclose(a[:, 0], t)
    out = {k: a[:, col[k]] for k in OBS}
    # A focus is h2ax~foc at locus i in either state: Damage_Foci_i (not yet bound by ATM) or
    # Complete_Damage_Foci_i (bound); the two observables are mutually exclusive.
    foci = np.stack([a[:, col[f"Damage_Foci_{i}"]] + a[:, col[f"Complete_Damage_Foci_{i}"]]
                     for i in range(1, 51)])
    assert set(np.unique(foci)) <= {0.0, 1.0}, "one DNA molecule per locus, one focus state"
    out["foci"] = foci.sum(axis=0)
    out["runs"] = focus_runs(foci)
    shutil.rmtree(d)
    return out


with ThreadPoolExecutor(WORKERS) as ex:
    CELLS = list(ex.map(cell, SEEDS))
E = {k: np.stack([c[k] for c in CELLS]) for k in OBS + ["foci"]}
W30 = t <= 1800
print(f"{N_CELLS} cells, seeds {SEEDS[0]}..{SEEDS[-1]}, 0-{T_END} min every {DT} min")
print(f"  mean over cells and 0-30 h: ROS {E['ROS'][:, W30].mean():.2f}, p53 {E['p53'][:, W30].mean():.1f}, "
      f"p21 {E['p21'][:, W30].mean():.2f}, damage foci {E['foci'][:, W30].mean():.3f}")

300 cells, seeds 20150001..20150300, 0-2400 min every 1 min
  mean over cells and 0-30 h: ROS 10.26, p53 117.5, p21 1.19, damage foci 1.188


## Level 1: exact theory on the full model

### p53 mRNA, an immigration-death process

S1 Table gives p53 mRNA exactly two reactions, synthesis from the single promoter `P()` at
`kp53mRNAsyn` and first-order decay at `kp53mRNAdeg`; nothing else in the model makes or consumes
it. Starting from `n0` copies, the count at time t is the binomial survivors of `n0` plus an
independent Poisson immigrant population, so

  E(t) = n0 e^{-mu t} + (lambda/mu)(1 - e^{-mu t}),  Var(t) = n0 e^{-mu t}(1 - e^{-mu t}) + (lambda/mu)(1 - e^{-mu t}),

with lambda = kp53mRNAsyn x P and mu = kp53mRNAdeg. The file seeds `n0 = 10 = lambda/mu`, so the
mean is 10 at every time while the variance grows from 0 to the Poisson value 10.

Three asserts, each with the sampling error it is judged against:

* at every 10-min output the ensemble mean is within `z*` exact standard errors `sqrt(Var(t)/N)` of
  E(t), where `z* = Phi^-1(1 - 0.01/2n)` over the n = 240 outputs (a family-wise 1% false-alarm
  rate; the outputs are positively correlated, so this is conservative);
* the per-cell time average over 0-40 h, averaged over cells, is within 3 standard errors of 10;
* the variance at 40 h is within 3 standard errors of Var(40 h), using the sampling variance of a
  sample variance from a Poisson(10) population, `(mu4 - sigma^4 (N-3)/(N-1))/N` with
  `mu4 = 10(1 + 3x10)`.

In [4]:
lam = P["kp53mRNAsyn"] * SEED_COUNT["P()"]
mu = P["kp53mRNAdeg"]
n0 = SEED_COUNT["p53_mRNA()"]
e = np.exp(-mu * t)
EX = n0 * e + lam / mu * (1 - e)
VX = n0 * e * (1 - e) + lam / mu * (1 - e)
X = E["p53_mRNA"]
grid = (t > 0) & (t % 10 == 0)
Z_MRNA = (X.mean(0) - EX)[grid] / np.sqrt(VX[grid] / N_CELLS)
Z_STAR = norm.isf(0.01 / (2 * grid.sum()))
print(f"p53 mRNA: E(t) = {EX.min():.3f}..{EX.max():.3f}; ensemble mean {X.mean(0)[grid].min():.2f}.."
      f"{X.mean(0)[grid].max():.2f} over {grid.sum()} outputs")
print(f"  max |z| over the outputs = {np.abs(Z_MRNA).max():.2f}   (tolerance z* = {Z_STAR:.2f})")
assert np.abs(Z_MRNA).max() < Z_STAR

xbar = (X[:, 1:] - EX[1:]).mean(axis=1)            # per-cell time average of X - E(t)
MRNA_POOL = (xbar.mean() + EX[1:].mean(), xbar.std(ddof=1) / np.sqrt(N_CELLS))
print(f"  pooled 0-40 h mean {MRNA_POOL[0]:.3f} +- {MRNA_POOL[1]:.3f} (1 SE) vs exact "
      f"{EX[1:].mean():.3f}: |diff| = {abs(xbar.mean()):.3f} (tolerance 3 SE = {3 * MRNA_POOL[1]:.3f})")
assert abs(xbar.mean()) < 3 * MRNA_POOL[1]

s2 = X[:, -1].var(ddof=1)
lamT = VX[-1]                                      # Poisson(10) to within e^{-mu T} = 6e-7
se_s2 = math.sqrt((lamT * (1 + 3 * lamT) - lamT**2 * (N_CELLS - 3) / (N_CELLS - 1)) / N_CELLS)
MRNA_VAR = (s2, VX[-1], se_s2)
print(f"  variance at {T_END:.0f} min {s2:.2f} vs exact {VX[-1]:.2f}: |diff| = {abs(s2 - VX[-1]):.2f} "
      f"(tolerance 3 SE = {3 * se_s2:.2f})")
assert abs(s2 - VX[-1]) < 3 * se_s2

p53 mRNA: E(t) = 10.000..10.000; ensemble mean 9.71..10.55 over 240 outputs
  max |z| over the outputs = 2.99   (tolerance z* = 4.10)
  pooled 0-40 h mean 10.076 +- 0.067 (1 SE) vs exact 10.000: |diff| = 0.076 (tolerance 3 SE = 0.202)
  variance at 2400 min 9.88 vs exact 10.00: |diff| = 0.12 (tolerance 3 SE = 2.51)


### ROS balance

S1 Table makes ROS at the constant rate `kROS` (from `I()`, one copy) and at `kROSgen` per
phospho-p38, and removes it at `kdROS` per molecule and by the two damage reactions
`DNA(site~ok) + ROS -> DNA(site~sdsb|cdsb)` at `(kdam1 + kdam2)` per intact locus. ROS turns over
in 1/kdROS = 0.2 min, so at every output

  E[ROS] = (kROS + kROSgen E[p38p]) / kdROS - c,   0 <= c <= (kdam1 + kdam2) x 50 / kdROS x E[ROS],

i.e. `Y = ROS - (kROSgen/kdROS) p38p` has mean `kROS/kdROS = 10` up to a relative correction of at
most 1.2e-4 (all fifty loci intact). The check takes each cell's time average of Y over 10-1800 min
(after the 0.2-min relaxation from the seeded ROS = 0), averages over cells, and asserts it lies
within 3 standard errors of 10 plus that bound plus the drift term `|dE[ROS]/dt| / kdROS`, which is
bounded from the ensemble's own hourly means.

In [5]:
A_ROS = P["kROS"] * SEED_COUNT["I()"] / P["kdROS"]
B_ROS = P["kROSgen"] / P["kdROS"]
EPS_DAM = (P["kdam1"] + P["kdam2"]) * 50 / P["kdROS"]
WY = (t >= 10) & (t <= 1800)
Y = (E["ROS"] - B_ROS * E["p38p"])[:, WY].mean(axis=1)
hourly = lambda A: np.array([A[:, (t >= max(60 * h, 10)) & (t < 60 * (h + 1))].mean() for h in range(30)])
H_ROS = hourly(E["ROS"])
drift = np.abs(np.diff(H_ROS)).max() / 60 / P["kdROS"]
bias = A_ROS * EPS_DAM + drift
ROS_ID = (Y.mean(), Y.std(ddof=1) / np.sqrt(N_CELLS), bias)
print(f"kROS/kdROS = {A_ROS:.3f}; damage-consumption bound {A_ROS * EPS_DAM:.1e}; drift bound {drift:.1e}")
print(f"mean over cells of <ROS - {B_ROS:.2f} p38p>(10-1800 min) = {Y.mean():.4f} +- {ROS_ID[1]:.4f} (1 SE)")
print(f"  |diff from {A_ROS:.0f}| = {abs(Y.mean() - A_ROS):.4f}   (tolerance 3 SE + bias = "
      f"{3 * ROS_ID[1] + bias:.4f})")
assert abs(Y.mean() - A_ROS) < 3 * ROS_ID[1] + bias

kROS/kdROS = 10.000; damage-consumption bound 1.2e-03; drift bound 3.6e-04
mean over cells of <ROS - 0.30 p38p>(10-1800 min) = 10.0026 +- 0.0043 (1 SE)
  |diff from 10| = 0.0026   (tolerance 3 SE + bias = 0.0145)


## Level 2: the paper's statements about non-irradiated cells

Quoted from Dolan et al. (2015), *Stochastic simulations of cell fate*, unless stated:

1. "In non-irradiated cells under normal conditions the abundance of ROS is approximately constant."
2. "In non-irradiated cells the repair machinery is able to keep the levels of damage under
   control; none of the simulated cells progressed into early senescence under normal background
   levels of ROS."
3. Fig. 4B, "p21/ROS feedback activated (rare event under normal conditions)".
4. "The induction of DNA damage subsequently increases p53, which, due to the feedback between p53
   and MDM2, becomes a transient pulse."
5. *Model parameterisation and validation*, on Fig. 2: "almost all foci in non-irradiated cells
   resolve in 24 hours".
6. "However, if the background ROS production was increased 10-fold, the number of induced DNA
   damage foci increased and 32% of cells entered into early senescence after 30 hours."

Statement 2 is an exact claim and is asserted as one. Statements 1 and 3-5 are qualitative; each is
asserted in the reading stated in its cell. Statement 6 is the one number the paper gives for
non-irradiated cells; it is the positive control that shows the senescence switch of statement 2
can fire at all.

### Statement 2: no cell becomes senescent
`Senescent_Counter` counts `Sen(State~sen)`, which is absorbing. Asserted: it is 0 at every output
of every cell over 0-30 h. If the per-cell probability of senescence within 30 h were q, all N cells
would stay normal with probability (1 - q)^N, so the assert has 95% power at q = 1 - 0.05^(1/N).
The file's own action runs 40 h; a cell that turns senescent between 30 and 40 h is printed but lies
outside the paper's statement, which is about its 30-h simulations.

In [6]:
sen30 = E["Senescent_Counter"][:, W30].max(axis=1)
sen40 = E["Senescent_Counter"].max(axis=1)
N_SEN30, N_SEN40 = int((sen30 > 0).sum()), int((sen40 > 0).sum())
Q95 = 1 - 0.05 ** (1 / N_CELLS)
print(f"senescent cells: {N_SEN30} of {N_CELLS} by 30 h, {N_SEN40} of {N_CELLS} by 40 h "
      f"(the paper: none; 95% power at a per-cell probability of {Q95:.1%})")
assert N_SEN30 == 0

senescent cells: 0 of 300 by 30 h, 1 of 300 by 40 h (the paper: none; 95% power at a per-cell probability of 1.0%)


### Statement 1: ROS is approximately constant; statement 3: the p21/ROS feedback is rare

**Reading of "approximately constant".** A single cell's ROS fluctuates as a Poisson variable about
`kROS/kdROS = 10`, with standard deviation sqrt(10) = 3.2 molecules. The population's ROS is taken
to be approximately constant if its hourly mean never moves from 10 by as much as that
single-cell standard deviation over the 30 h. The p21-driven route to more ROS (Fig. 4B) would
show up here as an upward drift.

**Reading of "rare".** The feedback is the p21 -> GADD45 -> p38 -> ROS loop, and the model's own
mark of "p21 high" is the threshold `p21 > 15` in `kplus()`, above which the senescence counter
starts to count. A cell has the feedback activated if its p21 exceeds that threshold at any time in
0-30 h. "Rare" is read as fewer than one cell in ten; the fraction and its Wilson 95% interval are
printed. The binomial standard error at N = 300 is under 1.8 points for any fraction below 10%.

In [7]:
DEV_ROS = np.abs(H_ROS - A_ROS).max()
print(f"hourly ensemble-mean ROS over 0-30 h: {H_ROS.min():.2f}..{H_ROS.max():.2f}; "
      f"max |mean - {A_ROS:.0f}| = {DEV_ROS:.3f} (tolerance sqrt({A_ROS:.0f}) = {math.sqrt(A_ROS):.2f})")
assert DEV_ROS < math.sqrt(A_ROS)


def wilson(k, n, z=1.96):
    p = k / n
    c = (p + z * z / (2 * n)) / (1 + z * z / n)
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / (1 + z * z / n)
    return c - h, c + h


p21max = E["p21"][:, W30].max(axis=1)
N_FB = int((p21max > P21_HIGH).sum())
F_FB = N_FB / N_CELLS
print(f"cells with p21 > {P21_HIGH:.0f} at some time in 0-30 h: {N_FB} of {N_CELLS} = {F_FB:.1%} "
      f"(Wilson 95% {wilson(N_FB, N_CELLS)[0]:.1%}..{wilson(N_FB, N_CELLS)[1]:.1%}; tolerance < 10%)")
print(f"  per-cell maximum p21 over 0-30 h: median {np.median(p21max):.0f}, max {p21max.max():.0f}")
assert F_FB < 0.10

hourly ensemble-mean ROS over 0-30 h: 9.98..10.78; max |mean - 10| = 0.783 (tolerance sqrt(10) = 3.16)
cells with p21 > 15 at some time in 0-30 h: 3 of 300 = 1.0% (Wilson 95% 0.3%..2.9%; tolerance < 10%)
  per-cell maximum p21 over 0-30 h: median 5, max 19


### Statement 5: almost all foci resolve within 24 hours

A focus is phosphorylated H2AX in the focus state at a locus, `DNAi(h2ax~foc)`, whether or not ATM
has joined it: the sum of the file's two mutually exclusive observables `Damage_Foci_i` (the focus
before ATM binds; `h2ax~foc` with no bond) and `Complete_Damage_Foci_i` (bound to ATM). The first
alone is only the minutes between focus formation and ATM binding. A focus's lifetime is the length
of a contiguous run of outputs at which it is present, sampled every minute. Sampling can only merge two foci separated by less than a minute or miss one shorter
than a minute; both make lifetimes look longer, so the resolved fraction below is a lower bound.
Every focus that appears by 960 min is followed for at least 24 h within the 40-h run, so for those
the 24-h outcome is observed, not censored. "Almost all" is read as at least 90%: asserted, at most
10% of those foci are still present 24 h after they appear, with at least 100 such foci.

In [8]:
RUNS = np.concatenate([c["runs"] for c in CELLS])
elig = RUNS[RUNS[:, 0] <= T_END - 1440]
N_FOCI = len(elig)
F_LONG = float((elig[:, 1] > 1440).mean())
print(f"{len(RUNS)} foci in {N_CELLS} cells; {N_FOCI} appear by {T_END - 1440} min")
print(f"  lifetime median {np.median(elig[:, 1]):.0f} min, 90th percentile "
      f"{np.percentile(elig[:, 1], 90):.0f} min, longest {elig[:, 1].max():.0f} min")
print(f"  still present 24 h after appearing: {F_LONG:.2%} (tolerance <= 10%)")
assert N_FOCI >= 100 and F_LONG <= 0.10

27298 foci in 300 cells; 7744 appear by 960 min
  lifetime median 25 min, 90th percentile 81 min, longest 357 min
  still present 24 h after appearing: 0.00% (tolerance <= 10%)


### Statement 4: DNA damage raises p53 in a transient pulse

Damage is timed by the onset of a focus (as defined for statement 5) in a cell that has had no
focus for the preceding 6 h, at least 12 h before the end of the run. For each onset the p53
trajectory over the next 12 h is taken relative to the cell's mean p53 over the 6 h to 1 h before
onset; the onsets of a cell are averaged first, so cells, not onsets, are the independent units of
the standard error. `z* = Phi^-1(1 - 0.01/72)` is one-sided and family-wise 1% over the 72 lags of
the 10-min grid, 10-720 min, which also covers choosing the peak lag from the data.

* *Increases*, asserted: at the peak lag the mean rise exceeds `z*` standard errors.
* *Transient*, measured and **not asserted**. The reading fixed in advance was that the mean falls
  from its peak to 12 h after onset by more than `z*` standard errors of the per-cell difference.
  The first execution of this notebook (identical seeds, with a focus counted as `Damage_Foci_i`
  only) did not meet it: a fall of 16.4 +- 6.7 molecules, z = 2.5, with 64% of the peak left at
  12 h. The statistic is confounded: at the file's damage rate, `(kdam1 + kdam2) x 10 x 50 = 6e-3`
  breaks per minute, a cell takes about four more breaks in the 12 h after an onset and each raises
  p53 again, so the ensemble average need not fall even when single pulses do. Changing the
  statistic after seeing that would be choosing the test to pass, so the fall is printed against
  its tolerance and statement 4's "transient" is left unverified here. A clean test needs onsets
  with no further damage for 12 h (about 1 window in 75 at this rate) and so a much larger
  ensemble.

In [9]:
PRE, GAP, POST = 360, 60, 720                     # minutes
ip, ig, iq = PRE // DT, GAP // DT, POST // DT     # in outputs
per_cell, n_onsets = [], 0
for f, p in zip(E["foci"], E["p53"]):
    on = np.flatnonzero((f[1:] > 0) & (f[:-1] == 0)) + 1
    ev = [s for s in on if s >= ip and s + iq < len(f) and f[s - ip:s].max() == 0]
    if ev:
        n_onsets += len(ev)
        per_cell.append(np.mean([p[s:s + iq + 1] - p[s - ip:s - ig].mean() for s in ev], axis=0))
DP53 = np.array(per_cell)
LAG = np.arange(iq + 1) * DT
m_dp, se_dp = DP53.mean(0), DP53.std(0, ddof=1) / np.sqrt(len(DP53))
g = (LAG > 0) & (LAG % 10 == 0)
Z_PULSE = norm.isf(0.01 / g.sum())
k = np.argmax(m_dp[g])
PEAK = (LAG[g][k], m_dp[g][k], se_dp[g][k])
print(f"{n_onsets} isolated focus onsets in {len(DP53)} cells; basal p53 {E['p53'][:, W30].mean():.1f}")
print(f"  peak mean rise {PEAK[1]:.1f} +- {PEAK[2]:.1f} molecules at +{PEAK[0]:.0f} min: "
      f"z = {PEAK[1] / PEAK[2]:.1f} (tolerance > {Z_PULSE:.2f})")
assert PEAK[1] / PEAK[2] > Z_PULSE

fall = DP53[:, g][:, k] - DP53[:, -1]               # per cell: peak-lag value minus the 12-h value
FALL = (fall.mean(), fall.std(ddof=1) / np.sqrt(len(fall)))
FALL_MET = FALL[0] / FALL[1] > Z_PULSE
print(f"  fall from the peak to +{POST} min: {FALL[0]:.1f} +- {FALL[1]:.1f}: z = {FALL[0] / FALL[1]:.1f} "
      f"(tolerance > {Z_PULSE:.2f}: {'met' if FALL_MET else 'NOT met'}; not asserted, see above); "
      f"{m_dp[-1] / PEAK[1]:.0%} of the peak left at +{POST} min")
kt = k + np.argmin(m_dp[g][k:])
print(f"  descriptive only (lag chosen from the data): the mean rise is lowest after the peak at "
      f"+{LAG[g][kt]:.0f} min, {m_dp[g][kt]:.1f} +- {se_dp[g][kt]:.1f} = {m_dp[g][kt] / PEAK[1]:.0%} of the peak")

88 isolated focus onsets in 81 cells; basal p53 117.5
  peak mean rise 51.3 +- 5.8 molecules at +90 min: z = 8.9 (tolerance > 3.64)
  fall from the peak to +720 min: 19.4 +- 8.2: z = 2.4 (tolerance > 3.64: NOT met; not asserted, see above); 62% of the peak left at +720 min
  descriptive only (lag chosen from the data): the mean rise is lowest after the peak at +390 min, 24.3 +- 5.6 = 47% of the peak


### Statement 6: ten-fold background ROS sends 32% of cells into early senescence by 30 h

The same file, the same action, with `setParameter("kROS", 10*kROS)` before it, run to the paper's
30 h (`t_end=>1800`, an output every 10 min). Seeds `20151001 ... 20151000 + N`, disjoint from the
baseline's. The fraction of cells senescent at 30 h is compared with the printed 32%.

**Tolerance.** Both numbers are binomial fractions. The paper does not state how many cells it ran
for this figure; its population statistics use "a minimum of 100 cell simulations", so it is taken
to have used 100. The tolerance is 3 combined standard errors,
`3 sqrt(0.32 x 0.68 x (1/N + 1/100))`; rounding of the printed 32% (0.5 points) is negligible beside
it. If the paper ran more cells the true tolerance is tighter, and the margin is printed. The ROS
balance of Level 1, now with mean 100, confirms that the parameter change took effect.

In [10]:
N_10X = 150
SEEDS_10X = [20151000 + i for i in range(1, N_10X + 1)]
T10 = np.arange(0, 1800 + 10, 10, dtype=float)


def cell_10x(seed):
    d, _, hdr, a = run_bng(seed, set_params=[("kROS", 10 * P["kROS"])], t_end=1800, n_steps=180)
    col = {h: i for i, h in enumerate(hdr)}
    assert np.allclose(a[:, 0], T10)
    out = {k: a[:, col[k]] for k in ("ROS", "p38p", "p21", "Senescent_Counter")}
    out["foci"] = sum(a[:, col[f"Damage_Foci_{i}"]] + a[:, col[f"Complete_Damage_Foci_{i}"]]
                      for i in range(1, 51))
    shutil.rmtree(d)
    return out


with ThreadPoolExecutor(WORKERS) as ex:
    C10 = list(ex.map(cell_10x, SEEDS_10X))
E10 = {k: np.stack([c[k] for c in C10]) for k in C10[0]}
Y10 = (E10["ROS"] - B_ROS * E10["p38p"])[:, T10 >= 10].mean(axis=1)
H10 = np.array([E10["ROS"][:, (T10 >= max(60 * h, 10)) & (T10 < 60 * (h + 1))].mean() for h in range(30)])
bias10 = 10 * A_ROS * EPS_DAM + np.abs(np.diff(H10)).max() / 60 / P["kdROS"]
se10 = Y10.std(ddof=1) / np.sqrt(N_10X)
print(f"10x kROS: <ROS - {B_ROS:.2f} p38p> = {Y10.mean():.3f} +- {se10:.3f} vs kROS/kdROS = {10 * A_ROS:.0f} "
      f"(tolerance 3 SE + bias = {3 * se10 + bias10:.3f}); damage foci {E10['foci'].mean():.2f} per cell "
      f"vs {E['foci'][:, W30].mean():.3f} at baseline")
assert abs(Y10.mean() - 10 * A_ROS) < 3 * se10 + bias10

N_SEN10 = int((E10["Senescent_Counter"][:, -1] > 0).sum())
F_SEN10 = N_SEN10 / N_10X
TOL_10X = 3 * math.sqrt(0.32 * 0.68 * (1 / N_10X + 1 / 100))
print(f"senescent at 30 h: {N_SEN10} of {N_10X} = {F_SEN10:.1%} (Wilson 95% "
      f"{wilson(N_SEN10, N_10X)[0]:.1%}..{wilson(N_SEN10, N_10X)[1]:.1%}); the paper: 32%")
print(f"  |diff| = {abs(F_SEN10 - 0.32):.3f} (tolerance {TOL_10X:.3f})")
assert abs(F_SEN10 - 0.32) <= TOL_10X

10x kROS: <ROS - 0.30 p38p> = 100.039 +- 0.065 vs kROS/kdROS = 100 (tolerance 3 SE + bias = 0.212); damage foci 10.37 per cell vs 1.188 at baseline
senescent at 30 h: 56 of 150 = 37.3% (Wilson 95% 30.0%..45.3%); the paper: 32%
  |diff| = 0.053 (tolerance 0.181)


## Verification figure, cache and clean-up

`verify_dolan2015.png` carries every assert above. `reference/baseline_ensemble.npz` caches the
per-cell ensembles (baseline at 10-min resolution, the 10x run as computed, and the per-focus
lifetimes) so the statistics can be recomputed without re-running about a hundred CPU-hours of NFsim.

In [11]:
BLUE, ORANGE, AQUA, INK, MUTED = "#2a78d6", "#eb6834", "#1baf7a", "#222222", "#8a8a85"
th = t / 60
fig, ax = plt.subplots(2, 4, figsize=(22, 10))

a = ax[0, 0]
a.plot(th, X.mean(0), color=BLUE, lw=1.2, label=f"BNG NFsim ensemble mean (N = {N_CELLS})")
a.plot(th[::150], EX[::150], "o", mfc="none", mec=INK, ms=7, label="exact immigration-death mean")
a.fill_between(th, EX - Z_STAR * np.sqrt(VX / N_CELLS), EX + Z_STAR * np.sqrt(VX / N_CELLS),
               color=MUTED, alpha=0.25, lw=0, label=f"tolerance +-z* SE (z* = {Z_STAR:.2f})")
a.set(title="Level 1: p53 mRNA vs exact theory", xlabel="time (h)", ylabel="p53 mRNA (molecules)",
      ylim=(8, 12))
a.legend(fontsize=8, loc="lower right")

a = ax[1, 0]
a.plot(th[grid], Z_MRNA, color=BLUE, lw=1)
a.axhspan(-Z_STAR, Z_STAR, color=MUTED, alpha=0.2, lw=0)
a.axhline(0, color=INK, lw=0.6)
a.set(title=f"residual z(t): max |z| {np.abs(Z_MRNA).max():.2f} (tol {Z_STAR:.2f})", xlabel="time (h)",
      ylabel="(mean - exact) / SE")

hh = np.arange(30) + 0.5
H_Y = hourly(E["ROS"] - B_ROS * E["p38p"])
a = ax[0, 1]
a.plot(hh, H_ROS, color=BLUE, lw=1.2, marker=".", label="hourly mean ROS")
a.plot(hh, H_Y, color=ORANGE, lw=1.2, marker=".", label=f"hourly mean ROS - {B_ROS:.1f} p38p")
a.plot(hh[::3], np.full(10, A_ROS), "o", mfc="none", mec=INK, ms=7, label="kROS/kdROS = 10")
a.axhspan(A_ROS - math.sqrt(A_ROS), A_ROS + math.sqrt(A_ROS), color=MUTED, alpha=0.2, lw=0,
          label="'approximately constant': +-sqrt(10)")
a.set(title="ROS balance (Level 1) and statement 1", xlabel="time (h)", ylabel="ROS (molecules)",
      ylim=(6, 14))
a.legend(fontsize=8, loc="lower right")

a = ax[1, 1]
a.plot(hh, H_ROS - A_ROS, color=BLUE, lw=1, marker=".", label="hourly mean ROS - 10")
a.plot(hh, H_Y - A_ROS, color=ORANGE, lw=1, marker=".", label="hourly mean (ROS - 0.3 p38p) - 10")
a.axhline(0, color=INK, lw=0.6)
a.set(title=f"pooled ROS - 0.3 p38p: {ROS_ID[0]:.3f} +- {ROS_ID[1]:.3f} (tol 3 SE + {ROS_ID[2]:.1e})\n"
            f"max hourly |ROS - 10| {DEV_ROS:.2f} (tol {math.sqrt(A_ROS):.2f})",
      xlabel="time (h)", ylabel="deviation (molecules)")
a.legend(fontsize=8)

a = ax[0, 2]
a.plot(LAG / 60, m_dp, color=BLUE, lw=1.2, label=f"mean over {len(DP53)} cells")
a.fill_between(LAG / 60, m_dp - se_dp, m_dp + se_dp, color=BLUE, alpha=0.2, lw=0, label="+-1 SE")
a.axhline(0, color=INK, lw=0.6)
a.plot([PEAK[0] / 60, POST / 60], [PEAK[1], m_dp[-1]], "o", mfc="none", mec=INK, ms=7,
       label="peak and 12-h values compared")
a.set(title=f"Statement 4: p53 after a focus appears\nrise z {PEAK[1] / PEAK[2]:.1f} (tol > {Z_PULSE:.2f}); "
            f"12-h fall z {FALL[0] / FALL[1]:.1f}, not asserted",
      xlabel="time after focus onset (h)", ylabel="p53 - pre-onset mean (molecules)")
a.legend(fontsize=8)

a = ax[1, 2]
age = np.unique(np.round(np.logspace(0, np.log10(1440), 200)))
surv = np.array([(elig[:, 1] > x).mean() for x in age])
a.step(age, surv, where="post", color=BLUE, lw=1.2, label=f"{N_FOCI} foci appearing by 16 h")
a.axvline(1440, color=MUTED, ls="--", lw=0.8, label="24 h")
a.axhline(0.10, color=MUTED, ls=":", lw=0.8, label="tolerance: <= 10% left at 24 h")
a.set(xscale="log", xlim=(1, 2000), ylim=(0, 1.02),
      title=f"Statement 5: focus survival\n{F_LONG:.2%} still present at 24 h (tol <= 10%)",
      xlabel="focus age (min, log scale)", ylabel="fraction still present")
a.legend(fontsize=8)

a = ax[0, 3]
f10 = (E10["Senescent_Counter"] > 0).mean(0)
a.plot(T10 / 60, f10, color=ORANGE, lw=1.4, label=f"10x background ROS (N = {N_10X})")
a.plot(th, (E["Senescent_Counter"] > 0).mean(0), color=BLUE, lw=1.4, label=f"normal ROS (N = {N_CELLS})")
a.errorbar([30], [0.32], yerr=[TOL_10X], fmt="o", mfc="none", mec=INK, ecolor=INK, capsize=4, ms=8,
           label="paper: 32% at 30 h (+- tolerance)")
a.set(title=f"Statements 2 and 6: senescent fraction\nnormal ROS {N_SEN30}/{N_CELLS} by 30 h (paper: none); "
            f"10x ROS {F_SEN10:.1%} (tol 32 +- {100 * TOL_10X:.1f}%)",
      xlabel="time (h)", ylabel="fraction of cells senescent", ylim=(-0.02, 0.6))
a.legend(fontsize=8, loc="upper left")

a = ax[1, 3]
bins = np.arange(-0.5, max(p21max.max(), P21_HIGH) + 2.5, 1)
a.hist(p21max, bins=bins, color=BLUE, alpha=0.85, label="per-cell maximum p21, 0-30 h")
a.axvline(P21_HIGH + 0.5, color=MUTED, ls="--", lw=0.8, label=f"kplus threshold p21 > {P21_HIGH:.0f}")
a.set(title=f"Statement 3: p21 > {P21_HIGH:.0f} (feedback on)\nin {N_FB}/{N_CELLS} = {F_FB:.1%} of cells (tol < 10%)",
      xlabel="maximum p21 (molecules)", ylabel="cells")
a.legend(fontsize=8)

fig.suptitle("Verification of models/p53_nhej_dolan2015 against Dolan et al. (2015): BNG2.pl NFsim on the "
             "unchanged file, non-irradiated cells", fontsize=13)
ok = verdict_footer(fig, [
    f"p53 mRNA vs exact: max|z| {np.abs(Z_MRNA).max():.2f} < {Z_STAR:.2f} PASS",
    f"ROS balance {ROS_ID[0]:.3f} vs 10 (3 SE {3 * ROS_ID[1]:.3f}) PASS",
    f"no senescence {N_SEN30}/{N_CELLS} PASS",
    f"ROS constant +-{DEV_ROS:.2f} < 3.16 PASS",
    f"feedback {F_FB:.1%} < 10% PASS",
    f"foci >24 h {F_LONG:.2%} <= 10% PASS",
    f"p53 rise z {PEAK[1] / PEAK[2]:.1f} > {Z_PULSE:.2f} PASS",
    f"10x ROS {F_SEN10:.1%} vs 32 +- {100 * TOL_10X:.1f}% PASS"],
    extra=f"each PASS is an assert in verify_dolan2015.ipynb. Not asserted: the 12-h fall of the p53 pulse, "
          f"z {FALL[0] / FALL[1]:.1f} against {Z_PULSE:.2f} ({'met' if FALL_MET else 'NOT met'}; confounded by later damage)")
fig.tight_layout(rect=(0, 0.05, 1, 0.96))
fig.savefig(HERE / "verify_dolan2015.png", dpi=130, bbox_inches="tight")
plt.close(fig)

sub = slice(None, None, 10)
np.savez_compressed(
    REFDIR / "baseline_ensemble.npz",
    seeds=np.array(SEEDS), t=t[sub], **{k: E[k][:, sub].astype(np.int16) for k in OBS + ["foci"]},
    focus_runs=RUNS.astype(np.int32), focus_runs_cols=np.array(["onset_min", "duration_min", "censored"]),
    seeds_10x=np.array(SEEDS_10X), t_10x=T10,
    **{f"x10_{k}": E10[k].astype(np.int16) for k in E10})
shutil.rmtree(TMP)
print("wrote verify_dolan2015.png and reference/baseline_ensemble.npz")

wrote verify_dolan2015.png and reference/baseline_ensemble.npz
